# PVNet model smoke test: 2020 data

This notebook has two paths: a quick single-sample check using pre-staged stores, followed by a one-week data-preparation and training smoke test. Its tracked smoke-test configs live in `Luke-ARC/configs`, avoiding reliance on ignored local `configs/` files. Set `PVNET_DATA_ROOT` to choose where local/ARC data is stored; without it, the notebook uses `Luke/data/to_gitignore` under this repo. On a fresh ARC account, run `python Luke-ARC/download_ukv_pilot.py` first, then run the notebook top-to-bottom. The small pre-staged single-sample check is skipped automatically when its three stores are absent.

The trained model is a software smoke test, not a forecast-skill result.

## 1. Locate the existing sample stores

In [1]:
import os
from pathlib import Path

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the PVNet repository root from the notebook working directory.")

# On ARC set PVNET_DATA_ROOT to a persistent project-data path, e.g. $DATA/pvnet.
# The default preserves the existing local Mac data layout.
DATA_ROOT = Path(os.environ.get("PVNET_DATA_ROOT", REPO_ROOT / "Luke/data/to_gitignore")).expanduser()
DIAGNOSTIC_DIR = DATA_ROOT / "public_bucket_diagnostic"
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
SATELLITE_STORE = DIAGNOSTIC_DIR / "20200621T1200Z_GSP199_8frame_context_sampler.zarr"
UKV_STORE = DIAGNOSTIC_DIR / "ukv_20200621_06/ukv_gsp199_20200621_06_sampler.zarr"
PVLIVE_STORE = DIAGNOSTIC_DIR / "pvlive_gsp199_20200621T1200Z_context_sampler.zarr"
DATA_CONFIG_SOURCE = REPO_ROOT / "Luke-ARC/configs/sampler_2020_smoke.yaml"
MODEL_CONFIG_PATH = REPO_ROOT / "Luke-ARC/configs/late_fusion_3h_smoke.yaml"

required_configs = [DATA_CONFIG_SOURCE, MODEL_CONFIG_PATH]
missing_configs = [p for p in required_configs if not p.exists()]
if missing_configs:
    raise FileNotFoundError("Missing required config(s):\n" + "\n".join(map(str, missing_configs)))

sample_stores = [SATELLITE_STORE, UKV_STORE, PVLIVE_STORE]
missing_sample_stores = [p for p in sample_stores if not p.exists()]
SAMPLE_STORES_PRESENT = not missing_sample_stores

print("Repository:", REPO_ROOT)
print("Data root:", DATA_ROOT.resolve())
if SAMPLE_STORES_PRESENT:
    print("Pre-staged one-sample stores are available:")
    for path in sample_stores:
        print("Found:", path)
else:
    print("Pre-staged one-sample stores are absent; cells 3–4 will be skipped.")
    print("The week workflow below will build new inputs from the source data.")


Repository: /Users/mans5211/GitHub-repos/PVNet
Sample time: 2020-06-21 12:00 UTC; GSP 199
Found: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/20200621T1200Z_GSP199_8frame_context_sampler.zarr
Found: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/ukv_20200621_06/ukv_gsp199_20200621_06_sampler.zarr
Found: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/pvlive_gsp199_20200621T1200Z_context_sampler.zarr


## 2. Point a temporary sampler configuration at the stores

This mirrors the windows used by the configured model: UKV from −2 to +2 hours, generation and solar context from −2 to +3 hours, and the existing satellite history. It writes a run-specific YAML beside the ignored diagnostic data; checked-in configs are untouched.

In [2]:
import yaml

with DATA_CONFIG_SOURCE.open() as f:
    data_config = yaml.safe_load(f)
input_data = data_config["input_data"]
input_data["nwp"] = {"ukv": input_data["nwp"]["ukv"]}

generation_cfg = input_data["generation"]
generation_cfg["zarr_path"] = str(PVLIVE_STORE.resolve())
generation_cfg["interval_start_minutes"] = -120
generation_cfg["interval_end_minutes"] = 180
generation_cfg["time_resolution_minutes"] = 30

ukv_cfg = input_data["nwp"]["ukv"]
ukv_cfg["zarr_path"] = str(UKV_STORE.resolve())
ukv_cfg["interval_start_minutes"] = -60
ukv_cfg["interval_end_minutes"] = 180
ukv_cfg["time_resolution_minutes"] = 60

input_data["satellite"]["zarr_path"] = str(SATELLITE_STORE.resolve())
solar_cfg = input_data["solar_position"]
solar_cfg["interval_start_minutes"] = -120
solar_cfg["interval_end_minutes"] = 180
solar_cfg["time_resolution_minutes"] = 30

RUN_CONFIG_PATH = DIAGNOSTIC_DIR / "pvnet_2020_model_smoke_configuration.yaml"
with RUN_CONFIG_PATH.open("w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)
print("Wrote temporary sampler config:", RUN_CONFIG_PATH)
print("Satellite channels:", len(input_data["satellite"]["channels"]))
print("UKV channels:", len(ukv_cfg["channels"]))

Wrote temporary sampler config: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/pvnet_2020_model_smoke_configuration.yaml
Satellite channels: 11
UKV channels: 17


## 3. Assemble and inspect one real PVNetDataset sample

In [3]:
if SAMPLE_STORES_PRESENT:
    import pandas as pd
    from ocf_data_sampler.torch_datasets.pvnet_dataset import PVNetDataset
    from pvnet.datamodule import collate_fn

    T0 = pd.Timestamp("2020-06-21 12:00:00")  # naive timestamps in these local stores are UTC
    dataset = PVNetDataset(
        str(RUN_CONFIG_PATH),
        time_periods=[(T0.isoformat(sep=" "), T0.isoformat(sep=" "))],
    )
    if len(dataset) != 1:
        raise ValueError(f"Expected exactly one sample at {T0}; dataset contains {len(dataset)}.")
    sample = dataset[0]
    batch = collate_fn([sample])

    print("Satellite:", tuple(batch["satellite_actual"].shape), " [batch, time, channels, y, x]")
    print("UKV:", tuple(batch["nwp"]["ukv"]["nwp"].shape), " [batch, time, channels, y, x]")
    print("Generation:", tuple(batch["generation"].shape))
    print("Solar elevation:", tuple(batch["solar_elevation"].shape))
    print("GSP ID:", batch["location_id"])
else:
    print("Skipping the quick pre-staged single-sample dataset check; its input stores are not present.")


Satellite: (1, 7, 11, 24, 24)  [batch, time, channels, y, x]
UKV: (1, 5, 17, 24, 24)  [batch, time, channels, y, x]
Generation: (1, 11)
Solar elevation: (1, 11)
GSP ID: tensor([199])


## 4. Run the configured model in inference mode

The model config has an illustrative location-ID mapping. This one-GSP diagnostic maps GSP 199 to embedding index 0, as in the January smoke test. Evaluation mode allows batch size one for layers such as BatchNorm.

In [4]:
if SAMPLE_STORES_PRESENT and "batch" in globals():
    import torch
    from hydra.utils import instantiate
    from omegaconf import OmegaConf
    from pvnet.utils import validate_batch_against_config

    model_config = OmegaConf.load(MODEL_CONFIG_PATH)
    model_config.model.location_id_mapping = {199: 0}
    lightning_model = instantiate(model_config)
    model = lightning_model.model.cpu().eval()
    validate_batch_against_config(batch, model)

    with torch.inference_mode():
        prediction = model(batch)

    print("Input batch passed model-config validation.")
    print("Prediction shape:", tuple(prediction.shape), " [batch, forecast steps, quantiles]")
    print("Forecast steps:", model.forecast_len)
    print("Quantiles:", model.output_quantiles)
    print("All prediction values finite:", bool(torch.isfinite(prediction).all()))
    print("Random-weight prediction range:", (float(prediction.min()), float(prediction.max())))
else:
    print("Skipping the quick model-forward check; no pre-staged sample batch is available. The week training smoke test follows below.")


Input batch passed model-config validation.
Prediction shape: (1, 6, 3)  [batch, forecast steps, quantiles]
Forecast steps: 6
Quantiles: [0.1, 0.5, 0.9]
All prediction values finite: True
Random-weight prediction range: (-0.003362013027071953, 0.28300750255584717)


## What this establishes

If the final cell completes, the 2020 satellite, UKV and PV_Live sample can be assembled and passed through the configured PVNet model's forward path. This is a compatibility milestone before attempting a larger multi-day training run. The returned values are not realistic forecasts because the model starts from random weights. The next meaningful step is training on multiple dates and evaluating on held-out dates; forecast skill requires substantially more data and training.

## Week-long 2020 training pilot (June 21–27 UTC)

The single-sample forward check passed. This next stage prepares one full week for a small training run: 30-minute sample times, a chronological nominal 80/20 split with a forecast-horizon purge at the boundary, and three CPU training epochs. June 21–27 is chosen because the existing 2020 pilot is from June 21 and the public satellite archive contains the 2020 year.

The current local stores are only short pilots, so this stage acquires the missing data. UKV requires 56 forecast cycles (Wholesale1 and Wholesale2: 112 GRIB files). At the two pilot file sizes, the raw transfer is estimated at about 28 GB. The download cell processes one cycle at a time, retains only the small crop, and supports resuming from saved crops or partial files. It will ask for your CEDA token using hidden input. The satellite Zarr is public and needs no gcloud login. Review the manifest before running the download cell; the training cells come after acquisition and checks.

### 5. Define the week, model windows and GSP crop

In [8]:
import numpy as np
import pandas as pd
import ocf_data_sampler
import yaml
from eccodes import codes_get, codes_grib_new_from_file, codes_release

WEEK_START_UTC = pd.Timestamp("2020-06-21T00:00:00Z")
WEEK_END_UTC = pd.Timestamp("2020-06-28T00:00:00Z")  # exclusive
SAMPLE_INTERVAL = pd.Timedelta(minutes=30)
sample_t0s = pd.date_range(WEEK_START_UTC, WEEK_END_UTC - SAMPLE_INTERVAL, freq=SAMPLE_INTERVAL)
if len(sample_t0s) != 336:
    raise ValueError(f"Expected 336 half-hour sample times, got {len(sample_t0s)}")

with DATA_CONFIG_SOURCE.open() as f:
    data_cfg = yaml.safe_load(f)["input_data"]
with MODEL_CONFIG_PATH.open() as f:
    model_cfg = yaml.safe_load(f)["model"]

# Align the UKV input window with the model's 2-hour history and 2-hour forecast window.
ukv_cfg = data_cfg["nwp"]["ukv"]
ukv_cfg["interval_start_minutes"] = -120
ukv_cfg["interval_end_minutes"] = 120
nwp_step_minutes = int(ukv_cfg["time_resolution_minutes"])
nwp_length = (int(model_cfg["nwp_history_minutes"]["ukv"]) + int(model_cfg["nwp_forecast_minutes"]["ukv"])) // nwp_step_minutes + 1
model_channels = list(ukv_cfg["channels"])
source_for_model_channel = {
    "t": "t", "dswrf": "sdswrf", "dlwrf": "sdlwrf", "hcc": "hcc",
    "mcc": "mcc", "lcc": "lcc", "sde": "sde", "r": "r", "vis": "vis",
    "si10": "si10", "wdir10": "wdir10", "prate": "prate", "hcct": "hcct",
    "cdcb": "cbh", "dpt": "dpt", "prmsl": "prmsl", "h": "h",
}
if model_channels != list(source_for_model_channel):
    raise ValueError(f"UKV channel mapping does not match model config: {model_channels}")

# Derive the 24x24 crop from a small UKV pilot file. On ARC, download it first with Luke-ARC/download_ukv_pilot.py.
UKV_DIAGNOSTIC_DIR = DIAGNOSTIC_DIR / "ukv_20200621_06"
PILOT_GRIB = UKV_DIAGNOSTIC_DIR / "202006210600_u1096_ng_umqv_Wholesale1.grib"
if not PILOT_GRIB.exists():
    raise FileNotFoundError(f"Expected the verified pilot GRIB at {PILOT_GRIB}")
location_csv = (Path(__import__("ocf_data_sampler").__file__).parent / "gsp_location_data_legacy_format" / "uk_gsp_locations_20250109.csv")
locations = pd.read_csv(location_csv, usecols=["gsp_id", "gsp_name", "x_osgb", "y_osgb"])
location = locations.loc[locations.gsp_id.eq(199)]
if len(location) != 1 or location.iloc[0].gsp_name != "MELK_1":
    raise LookupError("Could not resolve GSP 199 (MELK_1) from the OCF location table.")
gsp_x, gsp_y = float(location.iloc[0].x_osgb), float(location.iloc[0].y_osgb)

grid_info = None
with PILOT_GRIB.open("rb") as stream:
    while True:
        gid = codes_grib_new_from_file(stream)
        if gid is None:
            break
        try:
            if codes_get(gid, "shortName") == "t" and codes_get(gid, "level") == 1:
                grid_info = {
                    "nx": int(codes_get(gid, "Ni")), "ny": int(codes_get(gid, "Nj")),
                    "x_first": float(codes_get(gid, "X1InGridLengths")), "x_last": float(codes_get(gid, "X2InGridLengths")),
                    "y_first": float(codes_get(gid, "Y1InGridLengths")), "y_last": float(codes_get(gid, "Y2InGridLengths")),
                }
                break
        finally:
            codes_release(gid)
if grid_info is None:
    raise ValueError("Could not read UKV grid metadata from the pilot GRIB.")
ukv_x = np.linspace(grid_info["x_first"], grid_info["x_last"], grid_info["nx"])
ukv_y = np.linspace(grid_info["y_first"], grid_info["y_last"], grid_info["ny"])[::-1]
center_i, center_j = int(np.abs(ukv_x - gsp_x).argmin()), int(np.abs(ukv_y - gsp_y).argmin())
x_slice, y_slice = slice(center_i - 12, center_i + 12), slice(center_j - 12, center_j + 12)
if x_slice.start < 0 or y_slice.start < 0 or x_slice.stop > len(ukv_x) or y_slice.stop > len(ukv_y):
    raise ValueError("The GSP 199 24x24 crop extends beyond the UKV grid.")
if not np.allclose(np.diff(ukv_x), 2000) or not np.allclose(np.diff(ukv_y), 2000):
    raise ValueError("Expected 2 km UKV grid spacing.")

ceda_cycle_hours = 3
ukv_dropout_minutes = min(int(v) for v in ukv_cfg["dropout_timedeltas_minutes"])
available_at_t0 = sample_t0s + pd.Timedelta(minutes=ukv_dropout_minutes)
required_ukv_inits = pd.DatetimeIndex(available_at_t0.floor(f"{ceda_cycle_hours}h")).unique().sort_values()
rows = []
for init_time in required_ukv_inits:
    for wholesale in (1, 2):
        filename = f"{init_time:%Y%m%d%H%M}_u1096_ng_umqv_Wholesale{wholesale}.grib"
        date_path = init_time.strftime("%Y/%m/%d")
        rows.append({"init_time_utc": init_time, "filename": filename,
                     "url": f"https://dap.ceda.ac.uk/badc/ukmo-nwp/data/ukv-grib/{date_path}/{filename}?download=1"})
ukv_week_manifest = pd.DataFrame(rows)

split_index = round(0.8 * len(sample_t0s))
TRAIN_T0S = sample_t0s[:split_index]
VAL_T0S = sample_t0s[split_index:]
print(f"Week: {WEEK_START_UTC} through {WEEK_END_UTC} (end exclusive)")
print(f"Nominal sample times: {len(sample_t0s)} at 30-minute intervals")
print("The exact 80/20 split is finalized after checking which sample windows have complete satellite frames.")
print(f"UKV cycles: {len(required_ukv_inits)}; files: {len(ukv_week_manifest)}")
print(f"UKV cycles span {required_ukv_inits[0]} to {required_ukv_inits[-1]}")
pilot_sizes = [p.stat().st_size for p in UKV_DIAGNOSTIC_DIR.glob("202006210600_u1096_ng_umqv_Wholesale[12].grib")]
if len(pilot_sizes) == 2:
    estimate_gb = sum(pilot_sizes) * len(required_ukv_inits) / 1e9
    print(f"Estimated raw UKV transfer: {estimate_gb:.1f} GB (based on the verified June 21 06 UTC files)")
print(f"Disk free at data root: {__import__('shutil').disk_usage(DATA_ROOT).free / 1e9:.1f} GB")
print("Satellite source: gs://public-datasets-eumetsat-solar-forecasting/satellite/EUMETSAT/SEVIRI_RSS/v4/2020_nonhrv.zarr")
print("Current local satellite pilot has only 8 frames; the full-year public Zarr will supply the week.")
display(ukv_week_manifest.groupby(ukv_week_manifest.init_time_utc.dt.strftime("%Y-%m-%d")).agg(cycles=("init_time_utc", "nunique"), files=("filename", "count")))

Week: 2020-06-21 00:00:00+00:00 through 2020-06-28 00:00:00+00:00 (end exclusive)
Nominal sample times: 336 at 30-minute intervals
The nominal 80/20 split is finalized after checking satellite coverage; training starts whose forecast targets reach the validation period are then purged.
UKV cycles: 56; files: 112
UKV cycles span 2020-06-20 18:00:00+00:00 to 2020-06-27 15:00:00+00:00
Estimated raw UKV transfer: 28.3 GB (based on the verified June 21 06 UTC files)
Disk free now: 319.6 GB
Satellite source: gs://public-datasets-eumetsat-solar-forecasting/satellite/EUMETSAT/SEVIRI_RSS/v4/2020_nonhrv.zarr
Current local satellite pilot has only 8 frames; the full-year public Zarr will supply the week.


,cycles,files
init_time_utc,,
2020-06-20,2,4
2020-06-21,8,16
2020-06-22,8,16
2020-06-23,8,16
2020-06-24,8,16
2020-06-25,8,16
2020-06-26,8,16
2020-06-27,6,12


### 5a. Check public satellite coverage before the large UKV transfer

This opens only the public Zarr metadata and time coordinate. It reports missing 5-minute scans, groups adjacent missing scans into outages, and calculates which PVNet sample times remain valid. A gap does not automatically invalidate the whole week; the OCF sampler removes only sample windows that cross a gap.

For context, a contemporaneous [NOAA OSPO relay of EUMETSAT’s June 2020 service notice](https://www.ospo.noaa.gov/data/messages/2020/06/MSG_20200623_1634.html) lists a planned RSS interruption from 23 June 09:00 to 25 June 09:00 UTC. It says Meteosat-9 RSS imagery was to replace Meteosat-10 imagery and warns of a short outage when switching satellites. The archive gaps fall within this window; the 25 June gap is one hour before its scheduled end. The 24 June gap lasts 40 minutes between retained scans, longer than the notice's expected 15 minutes or slightly more, so the schedule is a plausible explanation but does not fully establish the cause or the source satellite for each scan.

In [9]:
import ocf_blosc2
import xarray as xr
from ocf_data_sampler.select import find_contiguous_t0_periods, fill_time_periods

PUBLIC_SATELLITE_ZARR = (
    "gs://public-datasets-eumetsat-solar-forecasting/satellite/"
    "EUMETSAT/SEVIRI_RSS/v4/2020_nonhrv.zarr"
)
satellite_metadata = xr.open_zarr(
    PUBLIC_SATELLITE_ZARR, consolidated=False, chunks=None,
    storage_options={"token": "anon"},
)
sat_time_name = "time" if "time" in satellite_metadata.coords else "time_utc"
available_satellite_times = pd.DatetimeIndex(
    pd.to_datetime(satellite_metadata[sat_time_name].values, utc=True)
).tz_localize(None)  # archive timestamps are UTC; OCF's local sampler coordinates are naive UTC
required_satellite_times = pd.date_range(
    WEEK_START_UTC.tz_localize(None) - pd.Timedelta(minutes=30),
    WEEK_END_UTC.tz_localize(None) - SAMPLE_INTERVAL,
    freq="5min",
)
missing_satellite_times = required_satellite_times.difference(available_satellite_times)
print("Public satellite archive time range:", available_satellite_times.min(), "to", available_satellite_times.max())
print("Required 5-minute scans:", len(required_satellite_times))
print("Missing required scans:", len(missing_satellite_times))

# Summarize each contiguous run of absent scans and the actual gap between retained frames.
outage_rows = []
for missing_time in missing_satellite_times:
    if not outage_rows or missing_time - outage_rows[-1]["last_missing_scan_utc"] > pd.Timedelta(minutes=5):
        previous = available_satellite_times[available_satellite_times < missing_time]
        outage_rows.append({
            "first_missing_scan_utc": missing_time,
            "last_missing_scan_utc": missing_time,
            "missing_scans": 1,
            "last_available_before_utc": previous.max() if len(previous) else pd.NaT,
        })
    else:
        outage_rows[-1]["last_missing_scan_utc"] = missing_time
        outage_rows[-1]["missing_scans"] += 1
for row in outage_rows:
    following = available_satellite_times[available_satellite_times > row["last_missing_scan_utc"]]
    row["first_available_after_utc"] = following.min() if len(following) else pd.NaT
    row["observed_gap_minutes"] = (
        row["first_available_after_utc"] - row["last_available_before_utc"]
    ).total_seconds() / 60 if pd.notna(row["first_available_after_utc"]) else np.nan
if outage_rows:
    display(pd.DataFrame(outage_rows))

# Use OCF's own continuity logic to find sample starts whose entire 7-frame input window exists.
satellite_periods = find_contiguous_t0_periods(
    available_satellite_times,
    interval_start=pd.Timedelta(minutes=-30),
    interval_end=pd.Timedelta(0),
    time_resolution=pd.Timedelta(minutes=5),
)
all_satellite_valid_t0s = fill_time_periods(satellite_periods, freq=SAMPLE_INTERVAL)
nominal_week_t0s = pd.date_range(
    WEEK_START_UTC.tz_localize(None),
    (WEEK_END_UTC - SAMPLE_INTERVAL).tz_localize(None),
    freq=SAMPLE_INTERVAL,
)
WEEK_VALID_T0S = nominal_week_t0s.intersection(all_satellite_valid_t0s)
DROPPED_SATELLITE_T0S = nominal_week_t0s.difference(WEEK_VALID_T0S)

# Make a chronological 80/20 split, then purge training origins whose
# three-hour target window reaches the first validation forecast origin.
split_index = round(0.8 * len(WEEK_VALID_T0S))
candidate_train_t0s = WEEK_VALID_T0S[:split_index]
VAL_T0S = WEEK_VALID_T0S[split_index:]
forecast_horizon_minutes = int(model_cfg["forecast_minutes"])
forecast_horizon = pd.Timedelta(minutes=forecast_horizon_minutes)
TRAIN_T0S = candidate_train_t0s[
    candidate_train_t0s + forecast_horizon < VAL_T0S[0]
]
PURGED_TRAIN_T0S = candidate_train_t0s.difference(TRAIN_T0S)
if not len(TRAIN_T0S) or not len(VAL_T0S):
    raise ValueError("The chronological split left an empty train or validation set.")
if TRAIN_T0S[-1] + forecast_horizon >= VAL_T0S[0]:
    raise ValueError("Training forecast targets reach into the validation period.")
print(f"Nominal sample starts: {len(nominal_week_t0s)}")
print(f"Valid sample starts after applying OCF satellite continuity rules: {len(WEEK_VALID_T0S)}")
print(f"Dropped sample starts: {len(DROPPED_SATELLITE_T0S)}")
if len(DROPPED_SATELLITE_T0S):
    print("Dropped t0s:", DROPPED_SATELLITE_T0S.tolist())
print(f"Purged {len(PURGED_TRAIN_T0S)} candidate training starts for a {forecast_horizon_minutes}-minute target embargo.")
print(f"Final chronological split: {len(TRAIN_T0S)} train ({len(TRAIN_T0S)/len(WEEK_VALID_T0S):.1%} of valid starts), {len(VAL_T0S)} validation ({len(VAL_T0S)/len(WEEK_VALID_T0S):.1%}).")
print(f"Train t0 range: {TRAIN_T0S[0]} to {TRAIN_T0S[-1]}")
print(f"Validation t0 range: {VAL_T0S[0]} to {VAL_T0S[-1]}")
print(f"Last training target horizon ends at {TRAIN_T0S[-1] + forecast_horizon}; validation starts at {VAL_T0S[0]}.")
satellite_metadata.close()
print("Satellite coverage check complete; no download has started.")

Public satellite archive time range: 2020-01-01 00:00:00 to 2020-12-31 23:55:00
Required 5-minute scans: 2017
Missing required scans: 8


,first_missing_scan_utc,last_missing_scan_utc,missing_scans,last_available_before_utc,first_available_after_utc,observed_gap_minutes
0,2020-06-24 08:00:00,2020-06-24 08:30:00,7,2020-06-24 07:55:00,2020-06-24 08:35:00,40.0
1,2020-06-25 08:00:00,2020-06-25 08:00:00,1,2020-06-25 07:55:00,2020-06-25 08:05:00,10.0


Nominal sample starts: 336
Valid sample starts after applying OCF satellite continuity rules: 331
Dropped sample starts: 5
Dropped t0s: [Timestamp('2020-06-24 08:00:00'), Timestamp('2020-06-24 08:30:00'), Timestamp('2020-06-24 09:00:00'), Timestamp('2020-06-25 08:00:00'), Timestamp('2020-06-25 08:30:00')]
Chronological split on available samples: 265 train (80.1%), 66 validation (19.9%)
Train t0 range: 2020-06-21 00:00:00 to 2020-06-26 14:30:00
Validation t0 range: 2020-06-26 15:00:00 to 2020-06-27 23:30:00
Satellite coverage check complete; no download has started.


Section 5a found eight missing 5-minute scans, not a long outage:
- 24 June, 08:00–08:30 UTC: seven consecutive scans are absent. The gap between the last available scan at 07:55 and the next at 08:35 is 40 minutes.
- 25 June, 08:00 UTC: one scan is absent; the surrounding available scans are 07:55 and 08:05.
Those dates fall within a planned RSS service interruption from 23 June 09:00 to 25 June 09:00. The notice said Meteosat-9 imagery would replace Meteosat-10 during the interruption and warned of gaps during satellite switches. That makes the interruption a plausible explanation, though the 40-minute gap is longer than the notice’s expected 15 minutes or slightly more, so we can’t confirm the precise cause from the archive timestamps alone. NOAA OSPO relay of the EUMETSAT notice

### 6. Download, crop and assemble the week's UKV data

For HTC, run `Luke-ARC/prepare_ukv_week.sbatch` from an HTC login node and skip the following notebook download cell. The batch worker downloads the 56 cycles, crops the 17 channels to 24×24 around GSP 199, and checkpoints crops so it can resume after interruption. It writes `ukv_gsp199_20200621_week_sampler.zarr` under `PVNET_DATA_ROOT/public_bucket_diagnostic/`.

The notebook cell below retains the same workflow for a local exploratory run. Do not run it on HTC after the batch job has produced the store.

In [11]:
import os
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from getpass import getpass
from pathlib import Path

import cfgrib
import numpy as np
import pandas as pd
import requests
import xarray as xr
from eccodes import codes_get, codes_grib_new_from_file, codes_release

if "ukv_week_manifest" not in globals() or "required_ukv_inits" not in globals():
    raise RuntimeError("Run the UKV week manifest cell first.")
if "grid_info" not in globals() or "x_slice" not in globals() or "y_slice" not in globals():
    raise RuntimeError("Run the GSP crop cell first so its grid and crop coordinates are defined.")
if "source_for_model_channel" not in globals() or "model_channels" not in globals():
    raise RuntimeError("Run the GSP crop cell first to define the UKV channel mapping.")

WEEK_RAW_DIR = DIAGNOSTIC_DIR / "ukv_20200621_28_raw"
WEEK_CROP_DIR = DIAGNOSTIC_DIR / "ukv_gsp199_20200621_week_cycles"
WEEK_ZARR = DIAGNOSTIC_DIR / "ukv_gsp199_20200621_week_sampler.zarr"
WEEK_RAW_DIR.mkdir(parents=True, exist_ok=True)
WEEK_CROP_DIR.mkdir(parents=True, exist_ok=True)
if WEEK_ZARR.exists():
    raise FileExistsError(f"Refusing to overwrite existing Zarr store: {WEEK_ZARR}")

ceda_token = os.environ.get("CEDA_TOKEN", "").strip() or getpass("Paste your CEDA access token (input is hidden): ").strip()
if not ceda_token:
    raise ValueError("No CEDA access token was entered.")

def retry_delay(response_headers, attempt):
    retry_after = response_headers.get("Retry-After")
    if retry_after:
        try:
            return max(1.0, float(retry_after))
        except ValueError:
            try:
                retry_time = parsedate_to_datetime(retry_after)
                if retry_time.tzinfo is None:
                    retry_time = retry_time.replace(tzinfo=timezone.utc)
                return max(1.0, (retry_time - datetime.now(timezone.utc)).total_seconds())
            except (TypeError, ValueError, OverflowError):
                pass
    return min(300.0, 15.0 * (2 ** (attempt - 1)))

def stream_ceda_file(url, destination, token, max_attempts=10):
    """Download atomically, resume partial bytes, and back off on CEDA throttling/errors."""
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + ".part")
    if destination.exists() and destination.stat().st_size > 0:
        return destination.stat().st_size, "already present"
    for attempt in range(1, max_attempts + 1):
        offset = partial.stat().st_size if partial.exists() else 0
        headers = {"Authorization": f"Bearer {token}"}
        if offset:
            headers["Range"] = f"bytes={offset}-"
        try:
            should_retry = False
            with requests.get(url, headers=headers, stream=True, timeout=(30, 180)) as response:
                if response.status_code == 429 or 500 <= response.status_code < 600:
                    if attempt == max_attempts:
                        response.raise_for_status()
                    retry_after_header = response.headers.get("Retry-After")
                    delay = retry_delay(response.headers, attempt)
                    print(f"CEDA HTTP {response.status_code} for {destination.name}; Retry-After={retry_after_header!r}; retry {attempt}/{max_attempts} in {delay:.0f}s.")
                    for header_name in ("X-RateLimit-Limit", "X-RateLimit-Remaining", "X-RateLimit-Reset"):
                        if response.headers.get(header_name) is not None:
                            print(f"  {header_name}: {response.headers[header_name]}")
                    should_retry = True
                else:
                    response.raise_for_status()
                    if "text/html" in response.headers.get("Content-Type", "").lower():
                        raise RuntimeError(f"CEDA returned HTML instead of GRIB for {destination.name}")
                    append = offset > 0 and response.status_code == 206
                    if offset and not append:
                        offset = 0  # Server ignored Range; restart this file cleanly.
                    mode = "ab" if append else "wb"
                    content_range = response.headers.get("Content-Range", "")
                    expected_total = (int(content_range.rsplit("/", 1)[1]) if "/" in content_range
                                      else int(response.headers["Content-Length"]) + offset
                                      if response.headers.get("Content-Length") else None)
                    with partial.open(mode) as output:
                        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                            if chunk:
                                output.write(chunk)
                    actual_size = partial.stat().st_size
                    if expected_total is not None and actual_size != expected_total:
                        raise IOError(f"Incomplete {destination.name}: got {actual_size:,} of {expected_total:,} bytes; partial retained for resume.")
                    partial.replace(destination)
                    return actual_size, "downloaded"
            if should_retry:
                time.sleep(delay)
                continue
        except requests.RequestException as error:
            status = error.response.status_code if isinstance(error, requests.HTTPError) and error.response is not None else None
            if status is not None and status != 429 and not 500 <= status < 600:
                raise
            if attempt == max_attempts:
                raise
            delay = min(300.0, 15.0 * (2 ** (attempt - 1)))
            print(f"Request failed for {destination.name} ({error}); retry {attempt}/{max_attempts} in {delay:.0f}s.")
            time.sleep(delay)
    raise RuntimeError(f"Failed to download {destination.name} after {max_attempts} attempts.")

def read_ukv_grid_metadata(grib_path):
    with Path(grib_path).open("rb") as stream:
        while True:
            gid = codes_grib_new_from_file(stream)
            if gid is None:
                break
            try:
                if codes_get(gid, "shortName") == "t" and codes_get(gid, "level") == 1:
                    return {"nx": int(codes_get(gid, "Ni")), "ny": int(codes_get(gid, "Nj")),
                            "x_first": float(codes_get(gid, "X1InGridLengths")),
                            "x_last": float(codes_get(gid, "X2InGridLengths")),
                            "y_first": float(codes_get(gid, "Y1InGridLengths")),
                            "y_last": float(codes_get(gid, "Y2InGridLengths"))}
            finally:
                codes_release(gid)
    raise ValueError(f"Could not read UKV grid metadata from {grib_path}")

def extract_cycle_crop(init_time, grib_paths):
    # Work out the valid times actually needed by the samples assigned to this init.
    t0s_for_init = sample_t0s[(available_at_t0.floor(f"{ceda_cycle_hours}h") == init_time)]
    nwp_cfg = data_cfg["nwp"]["ukv"]
    nwp_step_minutes = int(nwp_cfg["time_resolution_minutes"])
    nwp_length = (int(model_cfg["nwp_history_minutes"]["ukv"])
                  + int(model_cfg["nwp_forecast_minutes"]["ukv"])) // nwp_step_minutes + 1
    target_valid_times = set()
    for t0 in t0s_for_init:
        start = (t0 + pd.Timedelta(minutes=nwp_cfg["interval_start_minutes"])).ceil(f"{nwp_step_minutes}min")
        end = start + (nwp_length - 1) * pd.Timedelta(minutes=nwp_step_minutes)
        target_valid_times.update(pd.date_range(start, end, freq=f"{nwp_step_minutes}min"))
    target_valid_times = pd.DatetimeIndex(sorted(target_valid_times))
    steps = np.asarray([(valid_time - init_time).to_timedelta64() for valid_time in target_valid_times])
    if np.any(steps < np.timedelta64(0, "h")) or np.any(steps > np.timedelta64(36, "h")):
        raise ValueError(f"Required forecast leads outside 0–36 h for {init_time}: {steps}")

    current_grid = read_ukv_grid_metadata(grib_paths[0])
    if current_grid != grid_info:
        raise ValueError(f"UKV grid metadata changed at {init_time}: {current_grid}")
    values_by_channel = {}
    for grib_path in grib_paths:
        with xr.set_options(use_new_combine_kwarg_defaults=True):
            groups = cfgrib.open_datasets(str(grib_path), backend_kwargs={"indexpath": ""})
        try:
            for ds in groups:
                for channel, source_var in source_for_model_channel.items():
                    if channel in values_by_channel or source_var not in ds.data_vars:
                        continue
                    field = ds[source_var]
                    if channel == "t":
                        field = field.sel(heightAboveGround=1)
                    field = field.sel(step=steps)
                    values = np.asarray(field.values, dtype=np.float32)
                    expected = (len(steps), grid_info["nx"] * grid_info["ny"])
                    if values.shape != expected:
                        raise ValueError(f"Unexpected {source_var} shape in {grib_path.name}: {values.shape}; expected {expected}")
                    values = values.reshape(len(steps), grid_info["ny"], grid_info["nx"])[:, ::-1, :]
                    values_by_channel[channel] = values[:, y_slice, x_slice]
        finally:
            for ds in groups:
                ds.close()
    missing = [channel for channel in model_channels if channel not in values_by_channel]
    if missing:
        raise ValueError(f"Missing UKV channels for {init_time}: {missing}")

    # ecCodes decodes hcct's packed missing/no-cloud values into a range
    # of negative numbers around -32766 m, not only the exact sentinel.
    cycle_values = np.stack([values_by_channel[c] for c in model_channels], axis=1)
    cycle_values[:, model_channels.index("hcct")] = np.where(
        cycle_values[:, model_channels.index("hcct")] < 0, np.nan,
        cycle_values[:, model_channels.index("hcct")],
    )
    # OCF loader wants x/y dimensions and coordinates increasing in each direction.
    cycle_values = np.transpose(cycle_values, (0, 1, 3, 2))  # time, channel, x, y
    init_naive = init_time.tz_convert(None).to_datetime64()
    cycle_ds = xr.Dataset(
        {"UKV": (("init_time", "step", "variable", "x", "y"), cycle_values[None, ...])},
        coords={"init_time": [init_naive], "step": steps, "variable": model_channels,
                "x": ukv_x[x_slice], "y": ukv_y[y_slice]},
    )
    return cycle_ds

existing_manual_dir = DIAGNOSTIC_DIR / "ukv_20200621_06"
for cycle_number, init_time in enumerate(required_ukv_inits, start=1):
    crop_path = WEEK_CROP_DIR / f"ukv_gsp199_{init_time:%Y%m%dT%H%MZ}.nc"
    if crop_path.exists() and crop_path.stat().st_size > 0:
        print(f"[{cycle_number}/{len(required_ukv_inits)}] Reusing crop {crop_path.name}")
        continue
    cycle_rows = ukv_week_manifest.loc[ukv_week_manifest.init_time_utc.eq(init_time)]
    cycle_paths = []
    for row in cycle_rows.itertuples(index=False):
        manually_downloaded = existing_manual_dir / row.filename
        if manually_downloaded.exists():
            cycle_paths.append(manually_downloaded)
        else:
            raw_path = WEEK_RAW_DIR / row.filename
            size, state = stream_ceda_file(row.url, raw_path, ceda_token)
            print(f"  {state}: {row.filename} ({size / 1e6:.1f} MB)")
            time.sleep(3)  # spread requests to avoid repeatedly hitting CEDA's rate limit
            cycle_paths.append(raw_path)
    cycle_ds = extract_cycle_crop(init_time, cycle_paths)
    temporary_crop = crop_path.with_suffix(".nc.part")
    cycle_ds.to_netcdf(temporary_crop, engine="h5netcdf",
                       encoding={"UKV": {"zlib": True, "complevel": 2, "_FillValue": -9999.0}})
    temporary_crop.replace(crop_path)
    cycle_ds.close()
    for path in cycle_paths:
        if path.parent == WEEK_RAW_DIR:
            path.unlink(missing_ok=True)  # keep only the compact crop after successful processing
    print(f"[{cycle_number}/{len(required_ukv_inits)}] Saved {crop_path.name}; {cycle_ds.sizes['step']} valid times")

crop_paths = [WEEK_CROP_DIR / f"ukv_gsp199_{init_time:%Y%m%dT%H%MZ}.nc" for init_time in required_ukv_inits]
missing_crops = [path for path in crop_paths if not path.exists()]
if missing_crops:
    raise FileNotFoundError(f"Some cycle crops are missing: {missing_crops[:3]}")
cycle_datasets = [xr.open_dataset(path, engine="h5netcdf") for path in crop_paths]
try:
    week_ukv = xr.concat(cycle_datasets, dim="init_time").sortby("init_time")
    week_ukv.to_zarr(
        WEEK_ZARR, mode="w", consolidated=True, zarr_format=2,
        encoding={"UKV": {"_FillValue": np.nan}},
    )
finally:
    for ds in cycle_datasets:
        ds.close()
print(f"Wrote local GSP 199 UKV Zarr: {WEEK_ZARR.resolve()}")
print(week_ukv)

[1/56] Reusing crop ukv_gsp199_20200620T1800Z.nc
[2/56] Reusing crop ukv_gsp199_20200620T2100Z.nc
[3/56] Reusing crop ukv_gsp199_20200621T0000Z.nc
[4/56] Reusing crop ukv_gsp199_20200621T0300Z.nc
[5/56] Reusing crop ukv_gsp199_20200621T0600Z.nc
[6/56] Reusing crop ukv_gsp199_20200621T0900Z.nc
CEDA HTTP 500 for 202006211200_u1096_ng_umqv_Wholesale1.grib; Retry-After=None; retry 1/10 in 15s.
CEDA HTTP 500 for 202006211200_u1096_ng_umqv_Wholesale1.grib; Retry-After=None; retry 2/10 in 30s.
  downloaded: 202006211200_u1096_ng_umqv_Wholesale1.grib (252.4 MB)
  downloaded: 202006211200_u1096_ng_umqv_Wholesale2.grib (250.3 MB)
[7/56] Saved ukv_gsp199_20200621T1200Z.nc; 8 valid times
  downloaded: 202006211500_u1096_ng_umqv_Wholesale1.grib (251.9 MB)
  downloaded: 202006211500_u1096_ng_umqv_Wholesale2.grib (249.1 MB)
[8/56] Saved ukv_gsp199_20200621T1500Z.nc; 8 valid times
  downloaded: 202006211800_u1096_ng_umqv_Wholesale1.grib (252.0 MB)
  downloaded: 202006211800_u1096_ng_umqv_Wholesale2.gr

/var/folders/hr/8b35c81n5gs9czlq47x9dtk00000gp/T/ipykernel_7615/2331548166.py:224: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 2020-06-20T18:00:00'. Serializing with units 'hours since 2020-06-20T18:00:00' instead. Set encoding['dtype'] to floating point dtype to serialize with units 'days since 2020-06-20T18:00:00'. Set encoding['units'] to 'hours since 2020-06-20T18:00:00' to silence this warning .
  week_ukv.to_zarr(


### 7. Fetch PV_Live generation and required capacity metadata

The sampler requires both `generation_mw` and `capacity_mwp`. This cell queries the normal PV_Live API first, then tries Sheffield Solar’s backup endpoint if the normal endpoint returns missing/invalid historical capacity. If neither provides usable effective capacity, it falls back to the monthly installed-capacity proxy, clearly labelled as a plumbing-only substitute.

The selected response is checked for complete half-hour coverage before saving. The backup API is used only as a data-source fallback; the normal PV_Live URL remains primary.

In [23]:
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import xarray as xr
import ocf_data_sampler

GSP_ID = 199
PVLIVE_START = WEEK_START_UTC - pd.Timedelta(minutes=120)
PVLIVE_END = (WEEK_END_UTC - SAMPLE_INTERVAL) + pd.Timedelta(minutes=180)
query_params = {
    "start": PVLIVE_START.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "end": PVLIVE_END.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "extra_fields": "capacity_mwp",
}
api_roots = {
    "primary": "https://api.pvlive.uk",
    "backup": "https://api.solar.sheffield.ac.uk",
}

def fetch_pvlive(root):
    response = requests.get(
        f"{root}/pvlive/api/v4/gsp/{GSP_ID}",
        params=query_params,
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    frame = pd.DataFrame(payload["data"], columns=payload["meta"])
    required = {"datetime_gmt", "generation_mw"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{root} response is missing required columns: {frame.columns.tolist()}")
    frame["datetime_gmt"] = pd.to_datetime(frame["datetime_gmt"], utc=True)
    frame["generation_mw"] = pd.to_numeric(frame["generation_mw"], errors="coerce")
    if "capacity_mwp" in frame:
        frame["capacity_mwp"] = pd.to_numeric(frame["capacity_mwp"], errors="coerce")
    return frame.sort_values("datetime_gmt").drop_duplicates("datetime_gmt").reset_index(drop=True)

pvlive_week = fetch_pvlive(api_roots["primary"])
capacity_source = None
primary_capacity = (
    pvlive_week["capacity_mwp"].to_numpy(dtype=float)
    if "capacity_mwp" in pvlive_week else np.full(len(pvlive_week), np.nan)
)
if np.isfinite(primary_capacity).all() and (primary_capacity > 0).all():
    capacity_values = primary_capacity.astype(np.float32)
    capacity_source = "PV_Live primary outturn API capacity_mwp (effective capacity)"
    print("Using the normal PV_Live endpoint; capacity_mwp is complete.")
else:
    print("Normal PV_Live endpoint returned incomplete capacity_mwp; trying Sheffield Solar's backup endpoint.")
    try:
        backup_week = fetch_pvlive(api_roots["backup"])
        backup_capacity = (
            backup_week["capacity_mwp"].to_numpy(dtype=float)
            if "capacity_mwp" in backup_week else np.full(len(backup_week), np.nan)
        )
        if len(backup_capacity) and np.isfinite(backup_capacity).all() and (backup_capacity > 0).all():
            pvlive_week = backup_week
            capacity_values = backup_capacity.astype(np.float32)
            capacity_source = "Sheffield Solar backup outturn API capacity_mwp (effective capacity)"
            print("Backup endpoint returned complete capacity_mwp; using its generation and capacity response.")
        else:
            print("Backup endpoint also returned incomplete/invalid capacity_mwp; using the monthly installed-capacity proxy.")
    except (requests.RequestException, ValueError, KeyError) as exc:
        print(f"Backup endpoint request/response failed ({type(exc).__name__}: {exc}); using the monthly installed-capacity proxy.")

pvlive_week = pvlive_week.sort_values("datetime_gmt").drop_duplicates("datetime_gmt").reset_index(drop=True)
expected_times = pd.date_range(PVLIVE_START, PVLIVE_END, freq="30min")
actual_times = pd.DatetimeIndex(pvlive_week["datetime_gmt"])
missing_times = expected_times.difference(actual_times)
unexpected_times = actual_times.difference(expected_times)
print(f"Selected PV_Live rows: {len(pvlive_week)}; expected {len(expected_times)} half-hour timestamps.")
if len(missing_times) or len(unexpected_times) or len(actual_times) != len(expected_times):
    print("Missing timestamps (first 20):", missing_times[:20].tolist())
    print("Unexpected timestamps (first 20):", unexpected_times[:20].tolist())
    raise ValueError("PV_Live time coverage is incomplete or contains unexpected timestamps; no store was written.")
if not np.isfinite(pvlive_week["generation_mw"].to_numpy(dtype=float)).all():
    invalid = pvlive_week.loc[~np.isfinite(pvlive_week["generation_mw"].to_numpy(dtype=float)), ["datetime_gmt", "generation_mw"]]
    display(invalid.head(30))
    raise ValueError("PV_Live has missing/non-finite generation targets; no store was written.")

# If neither API supplied effective capacity, use the monthly installed-capacity
# report as a clearly labelled proxy for this software-plumbing smoke test.
if capacity_source is None:
    location_csv = Path(__import__("ocf_data_sampler").__file__).parent / "gsp_location_data_legacy_format" / "uk_gsp_locations_20250109.csv"
    locations = pd.read_csv(location_csv, usecols=["gsp_id", "gsp_name", "x_osgb", "y_osgb"])
    location_rows = locations.loc[locations.gsp_id.eq(GSP_ID)]
    if len(location_rows) != 1:
        raise LookupError(f"Could not resolve GSP {GSP_ID} in the OCF location table.")
    location = location_rows.iloc[0]

    index_response = requests.get("https://api.pvlive.uk/capacity/", timeout=60)
    index_response.raise_for_status()
    capacity_index = index_response.json()
    capacity_files = [
        (str(release), filename, url)
        for release, files in capacity_index.items()
        for filename, url in files.items()
        if filename.endswith("_GSP_and_month.csv.gz")
    ]
    if not capacity_files:
        raise LookupError("PV_Live capacity index has no monthly GSP capacity file.")
    capacity_release, capacity_filename, capacity_url = max(capacity_files)
    capacity_response = requests.get(capacity_url, timeout=120)
    capacity_response.raise_for_status()
    monthly_capacity = pd.read_csv(BytesIO(capacity_response.content), compression="gzip")
    required_columns = {"GSPs", "install_month", "cumul_capacity_mwp"}
    if not required_columns.issubset(monthly_capacity.columns):
        raise ValueError(f"Unexpected PV_Live capacity report columns: {monthly_capacity.columns.tolist()}")
    monthly_capacity = monthly_capacity.loc[monthly_capacity["GSPs"].eq(location.gsp_name)].copy()
    if monthly_capacity.empty:
        raise LookupError(f"GSP name {location.gsp_name!r} was not found in {capacity_filename}.")
    monthly_capacity["install_month"] = pd.to_datetime(monthly_capacity["install_month"])
    monthly_capacity["cumul_capacity_mwp"] = pd.to_numeric(monthly_capacity["cumul_capacity_mwp"], errors="coerce")
    monthly_capacity = monthly_capacity.sort_values("install_month").drop_duplicates("install_month", keep="last")
    sample_months = (
        pvlive_week["datetime_gmt"].dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()
        - pd.offsets.MonthBegin(1)
    )
    capacity_lookup = monthly_capacity.set_index("install_month")["cumul_capacity_mwp"]
    capacity_values = sample_months.map(capacity_lookup).to_numpy(dtype=float)
    if not np.isfinite(capacity_values).all() or (capacity_values <= 0).any():
        raise ValueError("Monthly PV_Live capacity report does not provide finite positive capacity for every sample timestamp.")
    capacity_values = capacity_values.astype(np.float32)
    capacity_source = (
        f"PV_Live monthly cumulative installed DC capacity proxy, lagged one month "
        f"(release {capacity_release}, {capacity_filename})"
    )
    print("Using monthly cumulative installed DC capacity, lagged one month; this is not effective capacity_mwp.")
    print("Capacity values used (MWp):", sorted(set(float(v) for v in capacity_values)))

from pyproj import Transformer
location_csv = Path(__import__("ocf_data_sampler").__file__).parent / "gsp_location_data_legacy_format" / "uk_gsp_locations_20250109.csv"
locations = pd.read_csv(location_csv, usecols=["gsp_id", "gsp_name", "x_osgb", "y_osgb"])
location_rows = locations.loc[locations.gsp_id.eq(GSP_ID)]
if len(location_rows) != 1:
    raise LookupError(f"Could not resolve GSP {GSP_ID} in the OCF location table.")
location = location_rows.iloc[0]
longitude, latitude = Transformer.from_crs(27700, 4326, always_xy=True).transform(float(location.x_osgb), float(location.y_osgb))
time_naive = pvlive_week["datetime_gmt"].dt.tz_localize(None).to_numpy()
generation_ds = xr.Dataset(
    data_vars={
        "generation_mw": (("time_utc", "location_id"), pvlive_week["generation_mw"].to_numpy(dtype=np.float32)[:, None]),
        "capacity_mwp": (("time_utc", "location_id"), capacity_values[:, None]),
    },
    coords={
        "time_utc": ("time_utc", time_naive),
        "location_id": ("location_id", np.array([GSP_ID], dtype=np.int32)),
        "longitude": ("location_id", np.array([longitude], dtype=np.float64)),
        "latitude": ("location_id", np.array([latitude], dtype=np.float64)),
    },
)
generation_ds["capacity_mwp"].attrs["source"] = capacity_source
generation_ds["capacity_mwp"].attrs["note"] = (
    "For a proxy, this is cumulative installed DC capacity from the prior month, not PV_Live effective capacity."
    if "proxy" in capacity_source else "PV_Live API capacity_mwp (effective capacity)."
)
PVLIVE_WEEK_STORE = DIAGNOSTIC_DIR / "pvlive_gsp199_20200621_week_sampler_backup_checked.zarr"
if PVLIVE_WEEK_STORE.exists():
    raise FileExistsError(f"Refusing to overwrite {PVLIVE_WEEK_STORE}; inspect or rename the existing store first.")
generation_ds.to_zarr(PVLIVE_WEEK_STORE, mode="w", consolidated=True, zarr_format=2)
print(f"Saved {len(pvlive_week)} validated PV_Live values with capacity metadata: {PVLIVE_WEEK_STORE}")
print(f"GSP {GSP_ID} ({location.gsp_name}) centre: {latitude:.5f}°N, {longitude:.5f}°E")
print(f"Capacity source: {capacity_source}")
print(f"Generation range: {pvlive_week.generation_mw.min():.3f}–{pvlive_week.generation_mw.max():.3f} MW")


Normal PV_Live endpoint returned incomplete capacity_mwp; trying Sheffield Solar's backup endpoint.
Backup endpoint returned complete capacity_mwp; using its generation and capacity response.
Selected PV_Live rows: 346; expected 346 half-hour timestamps.
Saved 346 validated PV_Live values with capacity metadata: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/pvlive_gsp199_20200621_week_sampler_backup_checked.zarr
GSP 199 (MELK_1) centre: 51.34409°N, -2.01115°E
Capacity source: Sheffield Solar backup outturn API capacity_mwp (effective capacity)
Generation range: 0.000–410.220 MW


### 8. Stage the satellite week locally from the public bucket

On HTC, run `Luke-ARC/stage_satellite_week.sbatch` from an HTC login node and skip the following notebook cell. It reads the public archive anonymously, selects the required June 2020 scans and the verified 24×24 GSP 199 crop, and writes a compact local Zarr. The eight archive gaps are preserved. Its 60-frame checkpoints let it resume after interruption.

The notebook cell below retains this workflow for a local exploratory run; do not run it on the NoMachine `nx-tsn01` kernel.


In [24]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import xarray as xr
from numcodecs import Blosc
from ocf_data_sampler.load.satellite import open_sat_data
from ocf_data_sampler.select import Location
from ocf_data_sampler.torch_datasets.utils import (
    add_alterate_coordinate_projections,
    slice_datasets_by_space,
)

if "required_satellite_times" not in globals() or "available_satellite_times" not in globals():
    raise RuntimeError("Run section 5a first to verify the public archive's week coverage.")
if "data_cfg" not in globals() or "DIAGNOSTIC_DIR" not in globals():
    raise RuntimeError("Run the setup cells through section 5a first.")

SATELLITE_WEEK_STORE = DIAGNOSTIC_DIR / "satellite_gsp199_20200621_week_public_sampler.zarr"
SATELLITE_WEEK_PARTS = DIAGNOSTIC_DIR / "satellite_gsp199_20200621_week_parts"
SATELLITE_WEEK_STORE.parent.mkdir(parents=True, exist_ok=True)
SATELLITE_WEEK_PARTS.mkdir(parents=True, exist_ok=True)
expected_satellite_times = pd.DatetimeIndex(required_satellite_times.intersection(available_satellite_times))
if not len(expected_satellite_times):
    raise ValueError("No required satellite scans are available for this week.")

if SATELLITE_WEEK_STORE.exists():
    staged = open_sat_data(str(SATELLITE_WEEK_STORE))
    try:
        got_times = pd.DatetimeIndex(staged.time_utc.values)
        if (staged.shape != (len(expected_satellite_times), 11, 24, 24)
                or not got_times.equals(expected_satellite_times)):
            raise ValueError(
                f"Existing staged satellite store is incomplete or mismatched: "
                f"shape={staged.shape}, expected={(len(expected_satellite_times), 11, 24, 24)}. "
                "Inspect or rename it before rerunning this cell."
            )
        print(f"Reusing validated local satellite Zarr: {SATELLITE_WEEK_STORE}")
        print(f"Frames: {len(got_times)}; archive gaps preserved: {len(required_satellite_times) - len(got_times)}")
    finally:
        staged.close()
else:
    satellite_cfg = dict(data_cfg["satellite"])
    public_ds = xr.open_zarr(
        PUBLIC_SATELLITE_ZARR,
        consolidated=False,
        chunks=None,
        storage_options={"token": "anon"},
    )
    try:
        data_vars = list(public_ds.data_vars)
        if len(data_vars) != 1:
            raise ValueError(f"Expected one satellite array in the public store; found {data_vars}.")
        public_da = public_ds[data_vars[0]]
        rename = {}
        if "time" in public_da.dims:
            rename["time"] = "time_utc"
        if "variable" in public_da.dims:
            rename["variable"] = "channel"
        public_da = public_da.rename(rename).sortby("time_utc").sortby("x_geostationary").sortby("y_geostationary")
        public_da = public_da.sel(channel=list(satellite_cfg["channels"]))

        # Match the centre and OCF crop used by the already-validated 2020 sample.
        gsp_location = Location(x=-2.0111, y=51.3441, coord_system="lon_lat", id=199)
        sat_cfg_for_crop = dict(satellite_cfg)
        sat_cfg_for_crop["zarr_path"] = str(SATELLITE_WEEK_STORE.resolve())
        crop_config = SimpleNamespace(
            input_data=SimpleNamespace(satellite=SimpleNamespace(**sat_cfg_for_crop))
        )
        add_alterate_coordinate_projections([gsp_location], {"sat": public_da})
        spatial_da = slice_datasets_by_space({"sat": public_da}, gsp_location, crop_config)["sat"]
        if spatial_da.sizes["x_geostationary"] != 24 or spatial_da.sizes["y_geostationary"] != 24:
            raise ValueError(f"Expected a 24x24 GSP crop, got {spatial_da.shape}.")

        reference_crop = DIAGNOSTIC_DIR / "20200621T1200Z_GSP199_8frame_context_sampler.zarr"
        if reference_crop.exists():
            reference = xr.open_zarr(reference_crop, consolidated=True)
            try:
                for coord in ("x_geostationary", "y_geostationary"):
                    if not np.allclose(spatial_da[coord].values, reference[coord].values, rtol=0, atol=1e-3):
                        raise ValueError(f"New {coord} crop differs from the previously validated GSP 199 crop.")
            finally:
                reference.close()

        source_times = pd.DatetimeIndex(pd.to_datetime(spatial_da.time_utc.values))
        missing_times = required_satellite_times.difference(source_times)
        selected_times = pd.DatetimeIndex(required_satellite_times.intersection(source_times))
        if not selected_times.equals(expected_satellite_times):
            raise ValueError("Satellite time selection differs from the coverage audit.")
        print(f"Required scans: {len(required_satellite_times)}; available: {len(selected_times)}; missing: {len(missing_times)}")
        if len(missing_times):
            print("Preserving missing scan times:", missing_times.tolist())
        print(f"Spatial crop: {spatial_da.sizes['x_geostationary']} × {spatial_da.sizes['y_geostationary']}; channels: {spatial_da.sizes['channel']}")

        area_attr = public_da.attrs.get("area")
        if area_attr is None:
            raise ValueError("Public satellite data has no area metadata.")
        batch_size = 60  # five-hour, independently checkpointed local stores
        batch_ranges = [
            (start, selected_times[start:start + batch_size])
            for start in range(0, len(selected_times), batch_size)
        ]
        for batch_number, (batch_start, batch_times) in enumerate(batch_ranges, start=1):
            part_store = SATELLITE_WEEK_PARTS / f"batch_{batch_number:03d}.zarr"
            part_staging = Path(str(part_store) + ".staging")
            if part_store.exists():
                part = xr.open_zarr(part_store, consolidated=True)
                try:
                    part_times = pd.DatetimeIndex(part.time.values)
                    if (part.sizes.get("time") != len(batch_times)
                            or not part_times.equals(batch_times)
                            or part["satellite_actual"].shape != (len(batch_times), 11, 24, 24)):
                        raise ValueError(f"Existing satellite checkpoint is mismatched: {part_store}")
                finally:
                    part.close()
                print(f"[{batch_number}/{len(batch_ranges)}] reusing checkpoint {part_store.name}")
                continue
            if part_staging.exists():
                raise FileExistsError(
                    f"Found an incomplete checkpoint at {part_staging}; inspect or rename it before continuing."
                )

            batch_da = spatial_da.sel(time_utc=batch_times.values).load()
            batch_ds = batch_da.rename({"time_utc": "time", "channel": "variable"}).to_dataset(name="satellite_actual")
            batch_ds.attrs["area"] = area_attr
            batch_ds.to_zarr(
                part_staging,
                mode="w",
                consolidated=True,
                zarr_format=2,
                encoding={
                    "satellite_actual": {
                        "compressor": Blosc(cname="lz4", clevel=5, shuffle=Blosc.SHUFFLE),
                        "chunks": (1, 11, 24, 24),
                    }
                },
            )
            batch_ds.close()
            part_staging.rename(part_store)
            print(f"[{batch_number}/{len(batch_ranges)}] saved {len(batch_times)} scans through {batch_times[-1]} UTC")

        if SATELLITE_WEEK_STORE.exists():
            raise FileExistsError(f"Final store appeared while staging: {SATELLITE_WEEK_STORE}")
        final_staging = Path(str(SATELLITE_WEEK_STORE) + ".staging")
        if final_staging.exists():
            raise FileExistsError(f"Found an incomplete final store at {final_staging}; inspect it before continuing.")
        parts = [xr.open_zarr(SATELLITE_WEEK_PARTS / f"batch_{n:03d}.zarr", consolidated=True)
                 for n in range(1, len(batch_ranges) + 1)]
        try:
            combined = xr.concat(parts, dim="time", data_vars="minimal", coords="minimal", compat="equals", join="exact")
            if not pd.DatetimeIndex(combined.time.values).equals(selected_times):
                raise ValueError("Satellite checkpoint concatenation produced unexpected times.")
            combined.to_zarr(
                final_staging,
                mode="w",
                consolidated=True,
                zarr_format=2,
                encoding={
                    "satellite_actual": {
                        "compressor": Blosc(cname="lz4", clevel=5, shuffle=Blosc.SHUFFLE),
                        "chunks": (1, 11, 24, 24),
                    }
                },
            )
        finally:
            for part in parts:
                part.close()
        final_staging.rename(SATELLITE_WEEK_STORE)
    finally:
        public_ds.close()

    staged = open_sat_data(str(SATELLITE_WEEK_STORE))
    try:
        staged_times = pd.DatetimeIndex(staged.time_utc.values)
        if staged.shape != (len(expected_satellite_times), 11, 24, 24) or not staged_times.equals(expected_satellite_times):
            raise ValueError(f"Saved satellite Zarr failed validation: shape={staged.shape}.")
        print(f"Validated local satellite Zarr: {staged.shape}; store={SATELLITE_WEEK_STORE}")
    finally:
        staged.close()


Reusing validated local satellite Zarr: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/satellite_gsp199_20200621_week_public_sampler.zarr
Frames: 2009; archive gaps preserved: 8


### 9. Configure the week and check dataset coverage

Satellite, UKV and PV_Live now use compact local stores prepared above. This cell builds the same PVNetDataset used by training, checks the chronological split after the forecast-horizon purge, and fetches one batch for shape/config validation before training.


In [25]:
import importlib.util
import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf
from pvnet.datamodule import PVNetDataModule
from pvnet.utils import validate_batch_against_config

if importlib.util.find_spec("ocf_blosc2") is None:
    raise ImportError("Install ocf-blosc2 in this kernel and restart it before opening the public satellite Zarr.")
import ocf_blosc2  # register the codec used by the public bucket

UKV_WEEK_STORE = DIAGNOSTIC_DIR / "ukv_gsp199_20200621_week_sampler.zarr"
if not UKV_WEEK_STORE.exists() or not PVLIVE_WEEK_STORE.exists():
    raise FileNotFoundError("Finish sections 6 and 7 before setting up the week dataset.")
week_data_cfg = yaml.safe_load(DATA_CONFIG_SOURCE.read_text())
week_input = week_data_cfg["input_data"]
week_input["nwp"] = {"ukv": week_input["nwp"]["ukv"]}
week_input["satellite"]["zarr_path"] = str(SATELLITE_WEEK_STORE.resolve())
week_input["generation"]["zarr_path"] = str(PVLIVE_WEEK_STORE.resolve())
week_input["generation"].update(interval_start_minutes=-120, interval_end_minutes=180, time_resolution_minutes=30)
week_input["nwp"]["ukv"]["zarr_path"] = str(UKV_WEEK_STORE.resolve())
week_input["nwp"]["ukv"].update(interval_start_minutes=-120, interval_end_minutes=120, time_resolution_minutes=60)
week_input["solar_position"].update(interval_start_minutes=-120, interval_end_minutes=180, time_resolution_minutes=30)
WEEK_CONFIG_PATH = DIAGNOSTIC_DIR / "pvnet_2020_week_smoke_configuration.yaml"
WEEK_CONFIG_PATH.write_text(yaml.safe_dump(week_data_cfg, sort_keys=False))

train_period = [(TRAIN_T0S[0].tz_localize(None).isoformat(sep=" "), TRAIN_T0S[-1].tz_localize(None).isoformat(sep=" "))]
val_period = [(VAL_T0S[0].tz_localize(None).isoformat(sep=" "), VAL_T0S[-1].tz_localize(None).isoformat(sep=" "))]
week_datamodule = PVNetDataModule(
    configuration=str(WEEK_CONFIG_PATH), train_periods=train_period, val_periods=val_period,
    batch_size=8, num_workers=0, prefetch_factor=None, persistent_workers=False, seed=2727831,
)
week_datamodule.setup("fit")
print("PVNetDataset counts:", len(week_datamodule.train_dataset), "train;", len(week_datamodule.val_dataset), "validation")
if len(week_datamodule.train_dataset) != len(TRAIN_T0S) or len(week_datamodule.val_dataset) != len(VAL_T0S):
    raise ValueError("Sampler coverage differs from the planned purged chronological split; inspect dataset valid_t0_times before training.")

model_config = OmegaConf.load(MODEL_CONFIG_PATH)
model_config.model.location_id_mapping = {199: 0}
week_lightning_model = instantiate(model_config)
train_batch = next(iter(week_datamodule.train_dataloader()))
validate_batch_against_config(train_batch, week_lightning_model.model)
print("Satellite batch:", tuple(train_batch["satellite_actual"].shape))
print("UKV batch:", tuple(train_batch["nwp"]["ukv"]["nwp"].shape))
print("Generation batch:", tuple(train_batch["generation"].shape))
print("Sample batch validated against the configured PVNet model.")

PVNetDataset counts: 265 train; 66 validation
Satellite batch: (8, 7, 11, 24, 24)
UKV batch: (8, 5, 17, 24, 24)
Generation batch: (8, 11)
Sample batch validated against the configured PVNet model.


### 10. Train for three CPU epochs and validate on the final ~20% (with forecast purge)

This is a small one-week training run, not a model-skill claim. It uses the purged chronological train/validation samples and saves no checkpoint.


In [26]:
from lightning.pytorch import Callback, Trainer

class WeekMetricHistory(Callback):
    """Keep per-epoch train/validation metrics visible in this notebook."""
    def __init__(self):
        self.rows = []

    def on_validation_epoch_end(self, trainer, pl_module):
        row = {"epoch": trainer.current_epoch + 1}
        for key, value in trainer.callback_metrics.items():
            if torch.is_tensor(value) and value.numel() == 1:
                row[key] = float(value.detach().cpu())
        self.rows.append(row)

metric_history = WeekMetricHistory()
# Keep the laptop default, but allow an HTC GPU job via PVNET_ACCELERATOR=gpu.
import os
RUN_ACCELERATOR = os.environ.get("PVNET_ACCELERATOR", "cpu").lower()
if RUN_ACCELERATOR == "gpu" and not torch.cuda.is_available():
    raise RuntimeError("PVNET_ACCELERATOR=gpu was requested, but this PyTorch environment cannot see a GPU.")
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "2")))
print(f"Training accelerator: {RUN_ACCELERATOR}")
week_trainer = Trainer(
    accelerator=RUN_ACCELERATOR, devices=1, max_epochs=3,
    logger=False, enable_checkpointing=False, enable_progress_bar=True, callbacks=[metric_history],
    enable_model_summary=False, log_every_n_steps=10,
    default_root_dir=str(DIAGNOSTIC_DIR / "training_run"),
)
week_trainer.fit(
    week_lightning_model,
    train_dataloaders=week_datamodule.train_dataloader(),
    val_dataloaders=week_datamodule.val_dataloader(),
)
print("Optimizer steps:", week_trainer.global_step)
print("Epochs completed:", week_trainer.current_epoch)
print("Metrics by epoch:")
display(pd.DataFrame(metric_history.rows))
print("Final metrics:")
for key, value in week_trainer.callback_metrics.items():
    if hasattr(value, "item"):
        value = value.item()
    print(f"  {key}: {value}")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_con
nector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_con
nector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


Optimizer steps: 99
Epochs completed: 3
Metrics by epoch:


,epoch,quantile_loss/val,MSE/val,MAE/val,val_fraction_below/fraction_below_0.10_quantile,val_fraction_below/fraction_below_0.50_quantile,val_fraction_below/fraction_below_0.90_quantile,val_step_MAE/step_000,val_step_MAE/step_001,val_step_MAE/step_002,...,MSE/val_persistence,quantile_loss/train,quantile_loss/train_step,MSE/train,MSE/train_step,MAE/train,MAE/train_step,quantile_loss/train_epoch,MSE/train_epoch,MAE/train_epoch
0,1,0.213895,0.052837,0.172231,0.301471,0.122243,0.076287,0.181331,0.205410,0.175505,...,0.015537,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.098864,0.043542,0.146351,0.011755,0.136158,0.557832,0.136695,0.192639,0.118918,...,0.015852,0.176595,0.176595,0.085990,0.085990,0.194306,0.194306,NaN,NaN,NaN
2,2,0.089894,0.061559,0.156704,0.000000,0.225897,0.769885,0.126075,0.121913,0.093168,...,0.015852,0.109471,0.109471,0.038295,0.038295,0.130778,0.130778,0.225259,0.120298,0.255307
3,3,0.055277,0.018081,0.079405,0.000000,0.363808,0.925272,0.082035,0.082731,0.057666,...,0.015852,0.067814,0.067814,0.032400,0.032400,0.101687,0.101687,0.135280,0.073322,0.185228


Final metrics:
  quantile_loss/train: 0.09518476575613022
  quantile_loss/train_step: 0.06781427562236786
  MSE/train: 0.04563314467668533
  MSE/train_step: 0.032400280237197876
  MAE/train: 0.14069613814353943
  MAE/train_step: 0.10168721526861191
  quantile_loss/val: 0.055277019739151
  MSE/val: 0.018080834299325943
  MAE/val: 0.07940523326396942
  val_fraction_below/fraction_below_0.10_quantile: 0.0
  val_fraction_below/fraction_below_0.50_quantile: 0.36380794644355774
  val_fraction_below/fraction_below_0.90_quantile: 0.9252715706825256
  val_step_MAE/step_000: 0.0820351392030716
  val_step_MAE/step_001: 0.08273094147443771
  val_step_MAE/step_002: 0.057665709406137466
  val_step_MAE/step_003: 0.14230242371559143
  val_step_MAE/step_004: 0.07647491991519928
  val_step_MAE/step_005: 0.035222239792346954
  val_step_MSE/step_000: 0.01488551963120699
  val_step_MSE/step_001: 0.018794961273670197
  val_step_MSE/step_002: 0.008328057825565338
  val_step_MSE/step_003: 0.04436219483613968
